# 🌾 FarmSense — Notebook Kaggle
## Gemma 4 Good Hackathon | Kaggle × Google DeepMind

---

### Comment fonctionne ce notebook ?

Ce notebook fait exactement 5 choses, dans l'ordre :

1. **Installe les dépendances Python** (Gradio, gTTS, etc.)
2. **Installe Ollama** — le moteur qui fait tourner Gemma 4 localement sur le GPU Kaggle
3. **Télécharge le modèle Gemma 4** (12B si GPU disponible, 4B sinon)
4. **Clone le code FarmSense** depuis GitHub
5. **Lance l'application** → un lien public apparaît dans l'output

> ⚠️ **Important** : Dans les paramètres du notebook Kaggle, activez **GPU T4 x2** sous *Accelerator*.
> Sans GPU, le modèle sera très lent.


In [ ]:
# ════════════════════════════════════════════════════════
# ÉTAPE 1 — Dépendances Python
# ════════════════════════════════════════════════════════
# On installe les bibliothèques nécessaires au projet.
# Le -q veut dire "quiet" (silencieux) pour ne pas
# afficher des dizaines de lignes de logs inutiles.

!pip install -q gradio==4.44.0 gtts==2.5.1 Pillow requests
print('✅ Dépendances installées')

In [ ]:
# ════════════════════════════════════════════════════════
# ÉTAPE 2 — Installer Ollama et le démarrer
# ════════════════════════════════════════════════════════
# Ollama est un programme qui permet de faire tourner
# des modèles d'IA (comme Gemma 4) directement sur le
# serveur Kaggle, sans envoyer les données sur internet.

import subprocess, time, requests

print('Installation de Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh 2>/dev/null

# On lance Ollama en arrière-plan (comme un service)
print('Démarrage du serveur Ollama en arrière-plan...')
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# On attend qu'il soit prêt (quelques secondes)
for i in range(10):
    time.sleep(3)
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        if r.status_code == 200:
            print('✅ Ollama est prêt')
            break
    except:
        print(f'  Attente... ({(i+1)*3}s)')
else:
    print('⚠️ Ollama met du temps à démarrer. Réexécutez cette cellule si nécessaire.')

In [ ]:
# ════════════════════════════════════════════════════════
# ÉTAPE 3 — Choisir et télécharger Gemma 4
# ════════════════════════════════════════════════════════
# On détecte automatiquement si un GPU est disponible
# pour choisir la bonne taille de modèle.
#
#   GPU T4 (16GB)  → gemma4:12b  (meilleure qualité)
#   CPU seulement  → gemma4:4b   (plus rapide, moins précis)

import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)

if result.returncode == 0:
    vram = result.stdout.strip()
    print(f'GPU détecté — VRAM disponible : {vram}')
    MODEL = 'gemma4:12b'
else:
    print('Aucun GPU détecté — utilisation du mode CPU')
    MODEL = 'gemma4:4b'

print(f'Modèle sélectionné : {MODEL}')
print('Téléchargement en cours... (5 à 15 minutes selon la connexion)')
print('Vous pouvez voir la progression ci-dessous :')

!ollama pull {MODEL}
print(f'\n✅ {MODEL} téléchargé et prêt !')

In [ ]:
# ════════════════════════════════════════════════════════
# ÉTAPE 4 — Récupérer le code FarmSense depuis GitHub
# ════════════════════════════════════════════════════════
# On clone (= télécharge) le dépôt GitHub qui contient
# tout le code du projet.
#
# ⚠️ REMPLACEZ l'URL ci-dessous par celle de votre repo !

import os

GITHUB_URL = 'https://github.com/TON_USERNAME/farmsense'  # ← À MODIFIER

if not os.path.exists('farmsense'):
    !git clone {GITHUB_URL} farmsense
    print('✅ Code cloné depuis GitHub')
else:
    !git -C farmsense pull
    print('✅ Code mis à jour depuis GitHub')

# Vérifier la structure
!find farmsense -type f -name '*.py' -o -name '*.json' | sort

In [ ]:
# ════════════════════════════════════════════════════════
# ÉTAPE 5 — Lancer FarmSense
# ════════════════════════════════════════════════════════
# On configure le modèle choisi à l'étape 3,
# puis on lance l'application.
#
# Après quelques secondes, vous verrez apparaître :
#   Running on public URL: https://xxxx.gradio.live
#
# Ce lien est votre démo — copiez-le pour la soumission Kaggle !

import os, sys
os.environ['GEMMA_MODEL'] = MODEL

# On ajoute le dossier app/ au chemin Python
sys.path.insert(0, 'farmsense/app')
os.chdir('farmsense/app')

print(f'Lancement de FarmSense avec {MODEL}...')
print('⏳ Le lien public apparaîtra dans quelques secondes ci-dessous')
print('-' * 60)

!python app.py

---
## 🧪 Test rapide (optionnel)

Si vous voulez tester Gemma 4 directement sans passer par l'interface,
exécutez la cellule ci-dessous :

In [ ]:
# Test direct — envoie une question à Gemma 4 et affiche la réponse
import requests, json

response = requests.post(
    'http://localhost:11434/api/chat',
    json={
        'model': MODEL,
        'messages': [{
            'role':    'user',
            'content': 'Mon mil a des taches jaunes sur les feuilles depuis le bas. Que faire ?'
        }],
        'stream': False
    },
    timeout=60
)

print(response.json()['message']['content'])